# 04 Buyer replay

**Data:** chosen in `common.py` (`USE_FAKE_DATA`). Chart titles say whether it is real or fake.

A buyer picks **K styles every 4 weeks** to stock more of, then sees how they did over the next 4 weeks. Three buyers are compared:

1. **Model buyer:** the K styles with the highest *predicted* growth (what the app shows).
2. **Copy-last-month buyer:** the K styles that rose most over the *last* 4 weeks.
3. **Everything:** the average of all styles (a reference line).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from common import load_predictions, RISING_THRESHOLD, DATA_LABEL, MAIN_MODEL

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)
MODEL, K = MAIN_MODEL, 10

df = load_predictions()
df = df[(df["model"] == MODEL) & (df["horizon"] == 4)].copy()

# What the copy-last-month buyer sees: growth over the last 4 weeks (recent_jump, from common.py).
df["last_month_growth"] = df["recent_jump"]
df = df.dropna(subset=["last_month_growth"])

# One decision every 4 weeks, so each set of picks is judged on its own 4 weeks.
first = df["origin"].min()
decision_dates = pd.date_range(first, df["origin"].max(), freq="28D")
print(f"{len(decision_dates)} decisions from {first.date()} to {decision_dates[-1].date()}, {K} picks each")

## Replay every decision

In [ ]:
def judge(picks):
    return {"hit rate (%)": 100 * (picks["actual_growth"] > RISING_THRESHOLD).mean(),
            "average growth": picks["actual_growth"].mean()}

rows = []
for date in decision_dates:
    week = df[df["origin"] == date]
    rows.append({
        "date": date,
        **{f"model {k}": v for k, v in judge(week.nlargest(K, "predicted_growth")).items()},
        **{f"copy {k}": v for k, v in judge(week.nlargest(K, "last_month_growth")).items()},
        **{f"everything {k}": v for k, v in judge(week).items()},
    })
replay = pd.DataFrame(rows).set_index("date")
replay.round(2).head()

## Summary

**Decisions won** = how many times the model buyer's picks grew more than the copy-last-month buyer's.

In [ ]:
won = (replay["model average growth"] > replay["copy average growth"]).sum()
summary = pd.DataFrame({
    "Model buyer": [replay["model hit rate (%)"].mean(), replay["model average growth"].mean()],
    "Copy-last-month buyer": [replay["copy hit rate (%)"].mean(), replay["copy average growth"].mean()],
    "Everything": [replay["everything hit rate (%)"].mean(), replay["everything average growth"].mean()],
}, index=["average hit rate (%)", "average growth of picks"]).round(2)
print(f"Model buyer won {won} of {len(replay)} decisions against the copy-last-month buyer")
summary

## Decision by decision

Average growth of each buyer's picks over the 4 weeks after each decision. Above 1.0 = the picks gained search interest.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(replay.index, replay["model average growth"], marker="o", label=f"Model buyer (top {K})")
ax.plot(replay.index, replay["copy average growth"], marker="o", label=f"Copy-last-month buyer (top {K})")
ax.plot(replay.index, replay["everything average growth"], linestyle="--", color="grey", label="All styles")
ax.axhline(1.0, color="black", linewidth=0.8)
ax.set_title(f"Buyer replay: growth of picks over the next 4 weeks ({DATA_LABEL})")
ax.set_xlabel("Decision date")
ax.set_ylabel("Average growth (1.0 = no change)")
ax.legend()
fig.savefig(FIGURES / "04_buyer_replay.png", dpi=200, bbox_inches="tight")
plt.show()

## What this shows

Real data, 137 styles, LightGBM, 10 picks every 4 weeks, threshold 1.2, February 2025 – September 2026.

1. The model buyer won **17 of 22** decisions against copying last month's top risers (9 of 12 in 2025).
2. Average hit rate: **56.8%** vs. 35.0% for the copy buyer and 20.7% for all styles. Median growth
   of picks: **1.27** vs. 1.12 and 1.03. Medians are used because the February 2026 decision
   (growth 2.64, during the spike) would distort an average.
3. Three of the five lost decisions were near-ties (0.03 or less apart). The clear losses were at
   the end of March 2026, while the spike was still building and chasing the fastest risers paid off
   (2.25 vs. 1.46), and in late June 2026 (0.85 vs. 0.73). After the spike (late April – late June
   2026) almost no picks rose for either buyer, but the model's picks fell less than the average
   style each time (e.g. 0.94 vs. 0.66 in May).